# Week 2: Week 1 pipeline with the chunking fix
**Only change from Week 1:** input chunks are 256 tokens (was 1,024) and are cut only between fact lines, never mid-fact. In Week 1 the writer hit its 2,048-token reply limit on every chunk and about 22% of facts never reached memory (see `week_2/docs/question_trace_v1.md`). The 2,048 write limit is kept: it matches the upstream Mem-α config. The reader's input+output guard is raised to 16,384 tokens (was 8,192) because fuller memory exceeded it; Qwen3-4B supports 32K.

## Original Week 1 description
**No paid inference, no API keys, no training.** Downloads need internet; inference runs in your Colab runtime.

Run in order on a fresh **GPU** runtime. Allow time for model downloads. Save a copy of this notebook to Drive first.
The output directory is mounted Drive by default. This notebook is a feasibility pilot, not a reproduction of published scores.

**Stages:** install → inspect GPU/data → test memory tools → build memories with Mem-α → unload it → answer with Qwen3-4B → export results.

Defaults: 4-bit NF4, one sequence at a time, no thinking, 256-token input chunks cut on whole fact lines, 2,048-token maximum write response; two full 6K contexts, five questions each. Nothing silently truncates benchmark contexts.

**What differs from upstream:** Transformers instead of vLLM; quantized weights; CPU MiniLM embeddings instead of paid embeddings; explicit core-memory enablement; project chunking; local Qwen3-4B full-memory reader instead of the external reader server. The writer retains upstream memory prompts, schemas, parser and tool execution, including one generation followed by tool execution per chunk. This is the project's adapted baseline—not “unmodified Mem-α.” Freeze these choices across future conditions.

**Validation status:** notebook structure, Python syntax and selected CPU logic checked during authoring. GPU execution must be verified in Colab; no experimental results are prefilled.


## 1. Install inference dependencies
Run once in a fresh runtime. **After installation, use Runtime > Restart session, then continue at step 2.** A restart is necessary when replacing an already imported bitsandbytes library. Do not install the upstream training requirements. PyTorch comes from Colab; its version is recorded with the run.

**CUDA 13.0 repair:** this revision uses `bitsandbytes==0.50.2`, whose Linux wheels support CUDA 13.0. The earlier `0.46.1` pin did not include that binary. If continuing in your existing notebook, change that pin before rerunning installation so it cannot downgrade the repair. No CUDA override or source compilation is needed for a supported GPU/runtime.

If you already completed installation and only need the repair, run `%pip install --upgrade --no-deps bitsandbytes==0.50.2` in a temporary cell, restart the session, then rerun steps 2 onward. Keep the same Drive run name if step 6 failed before any memory construction.

Reference: [bitsandbytes 0.50.2 installation support](https://huggingface.co/docs/bitsandbytes/v0.50.2/en/installation).


In [ ]:
import subprocess, sys
packages = [
    'transformers==4.51.3', 'accelerate==1.7.0', 'bitsandbytes==0.50.2',
    'sentence-transformers==4.1.0', 'datasets==3.6.0',
    'qwen-agent==0.0.29', 'openai==1.78.1', 'python-dotenv==1.1.0',
    'json_repair==0.51.0', 'rank-bm25==0.2.2', 'tiktoken==0.9.0',
    'nltk==3.9.1', 'rapidfuzz==3.13.0', 'pyyaml==6.0.2', 'scikit-learn>=1.5,<2'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Installation complete. Restart the session now, then continue at step 2. Do not rerun the old install cell.')
print('The OpenAI library is only an upstream import dependency; API clients are blocked below.')


## 2. Persistent output and configuration
Keep the default models for the first attempt. `WRITER_ID = "Qwen/Qwen3-1.7B"` is a fallback feasibility test only; it is not an RL-trained Mem-α experiment. Use a **new RUN_NAME** whenever settings change. Reusing a run name resumes compatible completed chunks and questions.

A checkpoint folder should have only one owner/runtime writing to it. Drive mount permission is for saving your results.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
RUN_NAME = 'week2_memalpha_4bit_chunk256_01'
OUTPUT = Path('/content/drive/MyDrive/CLMM_week2') / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
WRITER_ID = 'YuWangX/Memalpha-4B'
READER_ID = 'Qwen/Qwen3-4B'
EMBED_ID = 'sentence-transformers/all-MiniLM-L6-v2'
CHUNK_TOKENS = 256  # Week 1 used 1024; see header
WRITE_TOKENS = 2048
ANSWER_TOKENS = 128
MAX_INPUT_PLUS_OUTPUT = 16384  # Week 1 used 8192; MH memory exceeded it once facts stopped being lost
QUESTIONS_PER_CONTEXT = 5
SEED = 42
MAX_STAGE_MINUTES = 90  # Checked between generations; cannot interrupt an in-flight generation.
MEMALPHA_COMMIT = '9c0e1f6482f8ea7a322012a8155247966285946f'
MAB_COMMIT = '538026089d1a8a8eff05121d0db89b388f360eba'
print('Results:', OUTPUT)


## 3. Hardware and pinned upstream source
This stops if no GPU is attached. Select a GPU under Colab's runtime settings and reconnect if needed. It records the GPU actually assigned; Pro does not guarantee a specific GPU.

Before downloading models, a tiny NF4 layer tests the actual CUDA quantization and forward-pass kernels. It uses negligible memory compared with an LLM and fails early if the installed wheel cannot run on your GPU.


In [ ]:
import os, sys, subprocess, json, time, gc, hashlib, random, copy, importlib.util, ast, re, string
import numpy as np
import torch
from huggingface_hub import HfApi
assert torch.cuda.is_available(), 'No GPU: select a GPU runtime, then rerun.'
# Exercise the same NF4/double-quantization path before downloading/loading an LLM.
import importlib.metadata
import bitsandbytes as bnb
BNB_EXPECTED = '0.50.2'
if bnb.__version__ != BNB_EXPECTED:
    raise RuntimeError(
        f'Loaded bitsandbytes {bnb.__version__}; expected {BNB_EXPECTED}. '
        'Update the install cell, run it, restart the session, then resume at step 2.')
if os.environ.get('BNB_CUDA_VERSION'):
    raise RuntimeError(
        'BNB_CUDA_VERSION is set. Remove this override and restart so bitsandbytes '
        'uses the CUDA version supplied by PyTorch.')
print('Python:', sys.version.split()[0], '| torch:', torch.__version__,
      '| torch CUDA:', torch.version.cuda, '| bitsandbytes:', bnb.__version__,
      '| GPU:', torch.cuda.get_device_name(0))

def check_nf4_cuda():
    layer = inputs = outputs = None
    try:
        layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=torch.float16,
            compress_statistics=True, quant_type='nf4').half().to('cuda')
        inputs = torch.ones((2, 64), device='cuda', dtype=torch.float16)
        with torch.inference_mode():
            outputs = layer(inputs)
        torch.cuda.synchronize()
        assert torch.isfinite(outputs).all().item(), 'NF4 forward pass produced non-finite values.'
        print('NF4 CUDA quantization and forward-pass check passed.')
    except Exception as error:
        raise RuntimeError(
            'NF4 GPU compatibility check failed before model loading. '
            'Save the printed Python/torch/CUDA/bitsandbytes/GPU versions and this traceback. '
            'If you just upgraded bitsandbytes, restart the session first. '
            'This is not by itself evidence that the 4B model is too large.') from error
    finally:
        del layer, inputs, outputs
        gc.collect()
        torch.cuda.empty_cache()

check_nf4_cuda()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def save_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    temp.replace(path)

def sha(text):
    return hashlib.sha256(text.encode()).hexdigest()

def checkout(url, dest, revision):
    dest = Path(dest)
    if not dest.exists():
        subprocess.run(['git', 'clone', '--quiet', url, str(dest)], check=True)
    subprocess.run(['git', '-C', str(dest), 'checkout', '--quiet', revision], check=True)
    actual = subprocess.check_output(['git', '-C', str(dest), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == revision
    return dest

UPSTREAM = checkout('https://github.com/wangyu-ustc/Mem-alpha.git', '/content/Mem-alpha', MEMALPHA_COMMIT)
BENCHMARK = checkout('https://github.com/HUST-AI-HYZ/MemoryAgentBench.git', '/content/MemoryAgentBench', MAB_COMMIT)
sys.path.insert(0, str(UPSTREAM))
# Do not expose keys or environment variables. Fail if upstream unexpectedly tries a paid API.
import openai
def blocked_api(*args, **kwargs):
    raise RuntimeError('Paid API clients are disabled in this notebook.')
openai.OpenAI = blocked_api
openai.AzureOpenAI = blocked_api

settings = dict(writer=WRITER_ID, reader=READER_ID, embedding=EMBED_ID,
    chunk_tokens=CHUNK_TOKENS, write_tokens=WRITE_TOKENS, answer_tokens=ANSWER_TOKENS,
    max_input_plus_output=MAX_INPUT_PLUS_OUTPUT, questions=QUESTIONS_PER_CONTEXT,
    seed=SEED, memalpha_commit=MEMALPHA_COMMIT, benchmark_commit=MAB_COMMIT,
    quantization='NF4 double-quant float16 compute', including_core=True,
    writer_generation='greedy, thinking disabled', reader_mode='all stored memory',
    chunking='whole fact lines', notebook_protocol='week2-v1')
manifest_path = OUTPUT / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert manifest['settings'] == settings, 'Settings changed. Choose a new RUN_NAME.'
else:
    hub = HfApi()
    revisions = {m: hub.model_info(m).sha for m in set([WRITER_ID, READER_ID, EMBED_ID])}
    revisions['dataset'] = hub.dataset_info('ai-hyz/MemoryAgentBench').sha
    manifest = {'settings': settings, 'revisions': revisions, 'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    save_json(manifest_path, manifest)
revisions = manifest['revisions']
hardware = {'gpu': torch.cuda.get_device_name(0),
    'gpu_GiB': torch.cuda.get_device_properties(0).total_memory / 2**30,
    'torch': torch.__version__, 'python': sys.version, 'cuda': torch.version.cuda,
    'bitsandbytes': bnb.__version__, 'gpu_compute_capability': list(torch.cuda.get_device_capability(0)),
    'nf4_preflight': 'passed'}
save_json(OUTPUT / 'hardware.json', hardware)
(OUTPUT / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print(json.dumps(hardware, indent=2))


## 4. Load two complete published benchmark contexts
Uses the original MemoryAgentBench Conflict_Resolution split because these tasks need an explicit adapter to the Mem-α pipeline. Saves exact source IDs, context hashes, question indices and the dataset revision. The first five questions are a fixed **smoke subset**, not a representative test set. Do not tune your extension on these and later call them held-out results.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

ds = load_dataset('ai-hyz/MemoryAgentBench', revision=revisions['dataset'], split='Conflict_Resolution')
sources = [row['metadata']['source'] for row in ds]
print('Available sources:', sources)
targets = ['factconsolidation_sh_6k', 'factconsolidation_mh_6k']
selected = []
for source in targets:
    matches = [(i, row) for i, row in enumerate(ds) if row['metadata']['source'].lower() == source]
    assert len(matches) == 1, f'Expected one row for {source}; inspect the source list.'
    row_index, row = matches[0]
    assert len(row['questions']) == len(row['answers']) >= QUESTIONS_PER_CONTEXT
    assert all(isinstance(a, list) and a and all(isinstance(s, str) and s.strip() for s in a) for a in row['answers'])
    selected.append({'id': source, 'dataset_row': row_index, 'context': row['context'],
        'context_sha256': sha(row['context']), 'question_indices': list(range(QUESTIONS_PER_CONTEXT)),
        'questions': row['questions'][:QUESTIONS_PER_CONTEXT], 'answers': row['answers'][:QUESTIONS_PER_CONTEXT]})
save_json(OUTPUT / 'selected_examples.json', selected)
for item in selected:
    print(item['id'], 'characters:', len(item['context']), 'questions:', len(item['questions']))
    print('Context preview:', item['context'][:180])


## 5. Free embeddings and direct memory-tool checks (CPU)
MiniLM runs on CPU. Its 384-dimensional vectors replace the paid 1,536-dimensional embeddings, including the stored matrix dimensions. Long entries are embedded in 254-wordpiece windows and mean-pooled; no paid fallback or zero-vector fallback is used.

The direct insert/update/delete check tests infrastructure. It does **not** show that the language model chooses the right operations.


In [ ]:
from memory import Memory
from agent import MemoryAgent
from functions import get_memory_tool_schemas
import yaml
embedder = SentenceTransformer(EMBED_ID, revision=revisions[EMBED_ID], device='cpu')
embedding_stats = {'calls': 0, 'windows': 0}
class LocalMemory(Memory):
    def __init__(self):
        super().__init__(including_core=True)
        self.semantic_embedding_matrix = np.empty((0, 384), dtype=np.float32)
        self.episodic_embedding_matrix = np.empty((0, 384), dtype=np.float32)
    def _get_embedding(self, text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError('Cannot embed an empty or non-text memory.')
        ids = embedder.tokenizer.encode(text, add_special_tokens=False)
        windows = [embedder.tokenizer.decode(ids[i:i+254]) for i in range(0, len(ids), 254)]
        vectors = embedder.encode(windows, normalize_embeddings=True, show_progress_bar=False)
        vector = np.mean(vectors, axis=0).astype(np.float32)
        norm = np.linalg.norm(vector)
        assert np.isfinite(vector).all() and norm > 0
        embedding_stats['calls'] += 1; embedding_stats['windows'] += len(windows)
        return vector / norm

# Prevent accidental use of the upstream paid embedding implementation anywhere else.
Memory._get_embedding = blocked_api

def snapshot(memory):
    return copy.deepcopy({'core': memory.core, 'semantic': memory.semantic, 'episodic': memory.episodic})

def restore(state):
    memory = LocalMemory()
    memory.core = state['core']
    for kind in ['semantic', 'episodic']:
        setattr(memory, kind, copy.deepcopy(state[kind]))
        pairs = [(k, v) for entry in state[kind] for k, v in entry.items()]
        setattr(memory, kind + '_embedding_ids', [k for k, v in pairs])
        matrix = np.stack([memory._get_embedding(v) for k, v in pairs]) if pairs else np.empty((0, 384), dtype=np.float32)
        setattr(memory, kind + '_embedding_matrix', matrix)
    return memory

m = LocalMemory()
m.new_memory_insert('semantic', 'Ada lives in Paris.')
mid = next(iter(m.semantic[0]))
m.memory_update('semantic', 'Ada lives in Rome.', memory_id=mid)
assert m.semantic[0][mid] == 'Ada lives in Rome.'
assert m.semantic_embedding_matrix.shape == (1, 384)
assert m.memory_search('semantic', 'Where does Ada live?', search_method='text-embedding')
m.memory_delete('semantic', memory_id=mid)
assert not m.semantic and m.semantic_embedding_matrix.shape == (0, 384)
assert snapshot(restore(snapshot(m))) == snapshot(m)
save_json(OUTPUT / 'tool_checks.json', {'insert_update_delete_restore': 'passed', 'paid_api_clients': 'blocked'})
print('CPU memory-tool checks passed.')
# Create parser/tool helper without loading a model or invoking its constructor.
helper = MemoryAgent.__new__(MemoryAgent)
WRITE_TEMPLATE = yaml.safe_load((UPSTREAM / 'config/prompts_wrt_datasource.yaml').read_text())['unified_prompt']


## 6. Load one quantized model
Start with the defaults. An out-of-memory error is a feasibility finding: save it in the Week 1 report. Restart before retrying with changed settings. Do not repeatedly reload a model into a runtime that still holds the previous one.


In [ ]:
def load_lm(model_id):
    started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revisions[model_id])
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revisions[model_id],
        quantization_config=quant, device_map={'': 0}, torch_dtype=torch.float16,
        attn_implementation='sdpa').eval()
    print('Loaded', model_id, 'in', round(time.perf_counter()-started, 1), 'seconds')
    return model, tokenizer

def generate_text(model, tokenizer, prompt, max_new_tokens):
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    input_length = inputs['input_ids'].shape[1]
    assert input_length + max_new_tokens <= MAX_INPUT_PLUS_OUTPUT, (
        f'Prompt {input_length} + response {max_new_tokens} exceeds configured limit; no truncation performed.')
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    new_ids = output[0, input_length:]
    result = {'text': tokenizer.decode(new_ids, skip_special_tokens=True),
        'input_tokens': input_length, 'output_tokens': len(new_ids),
        'seconds': time.perf_counter()-started,
        'peak_allocated_GiB': torch.cuda.max_memory_allocated()/2**30,
        'peak_reserved_GiB': torch.cuda.max_memory_reserved()/2**30,
        'hit_output_limit': len(new_ids) >= max_new_tokens}
    del inputs, output, new_ids
    return result

writer, writer_tokenizer = load_lm(WRITER_ID)


## 7. Synthetic update test, then complete 6K contexts
The synthetic example explicitly separates an old fact, an unrelated fact, and a newer fact. It makes failures easy to inspect but is not benchmark evidence.

Each chunk saves its input, exact rendered prompt, raw response, parsed calls, tool results and memory before/after. Tool failures and output-length limits are logged; they are not treated as successful updates. An interrupted unfinished chunk is retried from the previous checkpoint. Changing settings requires a new run directory.


In [ ]:
def chunk_context(text, tokenizer, size):
    """Group whole lines (one fact per line) into chunks of at most ~size tokens.
    Week 1 cut every `size` tokens, which could split a fact across two chunks."""
    chunks, current, current_tokens = [], '', 0
    for line in text.splitlines(keepends=True):
        n = len(tokenizer(line, add_special_tokens=False)['input_ids'])
        if current and current_tokens + n > size:
            chunks.append(current)
            current, current_tokens = '', 0
        current += line
        current_tokens += n
    if current:
        chunks.append(current)
    assert ''.join(chunks) == text, 'Chunking altered the original text.'
    return chunks

toy = {'id': 'synthetic_update', 'context': '1. Ada lives in Paris.\n2. Ben likes chess.\n3. Ada now lives in Rome.',
    'chunks': ['1. Ada lives in Paris.', '2. Ben likes chess.', '3. Ada now lives in Rome.'],
    'questions': ['Where does Ada currently live?', 'What does Ben like?'],
    'answers': [['Rome'], ['chess']], 'question_indices': [0, 1]}
examples = [toy] + selected
save_json(OUTPUT / 'all_examples.json', examples)

def construct_memory(example, model, tokenizer, stage_started):
    folder = OUTPUT / example['id']; folder.mkdir(exist_ok=True)
    chunks = example.get('chunks') or chunk_context(example['context'], tokenizer, CHUNK_TOKENS)
    save_json(folder / 'chunks.json', chunks)
    memory = LocalMemory()
    events = []
    for index, chunk in enumerate(chunks):
        event_file = folder / f'chunk_{index:03d}.json'
        if event_file.exists():
            event = json.loads(event_file.read_text())
            assert event['input_chunk'] == chunk
            if events:
                assert event['before'] == events[-1]['after'], 'Checkpoint chain mismatch; use a fresh run.'
            events.append(event)
            continue
        # Restore only once, at the first unfinished chunk.
        if events and len(events) == index:
            memory = restore(events[-1]['after'])
        if (time.perf_counter() - stage_started)/60 > MAX_STAGE_MINUTES:
            raise TimeoutError('Stage time budget reached. Completed chunks are saved. Rerun this cell to resume.')
        started = time.perf_counter()
        before = snapshot(memory)
        text = WRITE_TEMPLATE.format(context=chunk, max_new_tokens=int(WRITE_TOKENS * .8))
        prompt = MemoryAgent.process_text_with_qwen_pipeline(text=text, tokenizer=tokenizer,
            functions=[t['function'] for t in get_memory_tool_schemas(memory)],
            status='memorie', enable_thinking=False, memory=memory, return_text=True)
        generation = generate_text(model, tokenizer, prompt, WRITE_TOKENS)
        messages = helper._parse_response(generation['text'])
        calls = []
        for message in messages:
            if message.get('function_call'):
                call = message['function_call']
                result = helper._run_tool_from_function_call(call, memory)
                calls.append({'call': call, 'result': result, 'error': ' error]' in result})
        event = {'index': index, 'input_chunk': chunk, 'rendered_prompt': prompt,
            'generation': generation, 'calls': calls, 'before': before, 'after': snapshot(memory),
            'wall_seconds': time.perf_counter()-started, 'no_tool_calls': not bool(calls)}
        save_json(event_file, event)
        events.append(event)
        print(example['id'], f'{index+1}/{len(chunks)}',
            'seconds:', round(event['wall_seconds']), 'tools:', len(calls),
            'errors:', sum(c['error'] for c in calls), 'output capped:', generation['hit_output_limit'])
    final_state = events[-1]['after']
    save_json(folder / 'memory_final.json', final_state)
    return events

stage_started = time.perf_counter()
for example in examples:
    construct_memory(example, writer, writer_tokenizer, stage_started)
print('Memory construction complete. Inspect synthetic_update/chunk_002.json before proceeding.')


## 8. Release the writer; load the fixed answering model
Only stored memory is sent to the reader for the main pilot questions. It does not see the original context or gold answers. All conditions in the later comparison must use this same reader configuration.


In [ ]:
if 'writer' in globals():
    del writer
if 'writer_tokenizer' in globals():
    del writer_tokenizer
gc.collect(); torch.cuda.empty_cache()
print('GPU allocated after release (GiB):', torch.cuda.memory_allocated()/2**30)
reader, reader_tokenizer = load_lm(READER_ID)


## 9. Reader controls and benchmark scoring
Loads normalization and matching functions directly from the pinned benchmark source. Reports both normalized exact match and substring exact match, using the best score over accepted aliases. Substring match can credit an answer containing both an old and new value: manually inspect such cases.

The control questions have their correct supporting facts supplied directly. They test whether the reader can follow simple supplied facts, not the difficulty of the full benchmark. A full-context check on the selected benchmark questions is also saved separately; it is a diagnostic upper-reference condition, not an oracle guarantee.


In [ ]:
# Import just the benchmark's pure scoring functions, avoiding its unrelated dependencies.
metric_names = {'normalize_answer', 'drqa_exact_match_score',
    'substring_exact_match_score', 'drqa_metric_max_over_ground_truths'}
metric_tree = ast.parse((BENCHMARK / 'utils/eval_other_utils.py').read_text())
metric_nodes = [n for n in metric_tree.body if isinstance(n, ast.FunctionDef) and n.name in metric_names]
assert {n.name for n in metric_nodes} == metric_names
metric_ns = {'re': re, 'string': string}
exec(compile(ast.Module(body=metric_nodes, type_ignores=[]), 'upstream_metrics', 'exec'), metric_ns)
def score(prediction, answers):
    return {name: float(metric_ns['drqa_metric_max_over_ground_truths'](metric_ns[fn], prediction, answers))
        for name, fn in [('exact_match', 'drqa_exact_match_score'), ('substring_exact_match', 'substring_exact_match_score')]}
assert score('The Rome.', ['Rome'])['exact_match'] == 1
assert score('Paris', ['Rome'])['substring_exact_match'] == 0

spec = importlib.util.spec_from_file_location('mab_templates', BENCHMARK / 'utils/templates.py')
templates = importlib.util.module_from_spec(spec); spec.loader.exec_module(templates)
query_template = templates.BASE_TEMPLATES['factconsolidation']['query']['long_context_agent']

def ask_reader(knowledge, question):
    messages = [{'role': 'system', 'content': 'Answer using only the supplied knowledge. If it does not contain the answer, say unknown.'},
        {'role': 'user', 'content': '[Knowledge Pool]\n' + knowledge + '\n\n' + query_template.format(question=question)}]
    prompt = reader_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    result = generate_text(reader, reader_tokenizer, prompt, ANSWER_TOKENS)
    result['rendered_prompt'] = prompt
    return result

controls = [
    ('1. Ada lives in Rome.', 'Where does Ada live?', ['Rome']),
    ('1. Ben likes chess.', 'What does Ben like?', ['chess']),
    ('1. Ada lives in Paris.\n2. Ada lives in Rome.', 'Where does Ada currently live?', ['Rome']),
    ('1. Mira works at Cedar Labs.\n2. Cedar Labs is in Oslo.', 'In which city does Mira work?', ['Oslo'])]
control_file = OUTPUT / 'reader_controls.json'
if control_file.exists():
    control_results = json.loads(control_file.read_text())
else:
    control_results = []
    for knowledge, question, answers in controls:
        result = ask_reader(knowledge, question)
        control_results.append({'question': question, 'answers': answers, 'generation': result, **score(result['text'], answers)})
    save_json(control_file, control_results)
print('Reader control exact matches:', sum(r['exact_match'] for r in control_results), '/', len(control_results))
if any(not r['exact_match'] for r in control_results):
    print('Inspect reader_controls.json. Reader errors may confound your memory experiment.')


In [ ]:
def memory_text(state):
    blocks = ['Core memory:\n' + (state['core'] or '')]
    for kind in ['semantic', 'episodic']:
        entries = [f'{key}: {value}' for entry in state[kind] for key, value in entry.items()]
        blocks.append(kind + ' memory:\n' + '\n'.join(entries))
    return '\n\n'.join(blocks)

stage_started = time.perf_counter()
for example in examples:
    folder = OUTPUT / example['id']
    state = json.loads((folder / 'memory_final.json').read_text())
    for condition, knowledge in [('stored_memory', memory_text(state)), ('full_context_control', example['context'])]:
        for qi, (question, answers) in enumerate(zip(example['questions'], example['answers'])):
            path = folder / f'answer_{condition}_{qi:03d}.json'
            if path.exists():
                continue
            if (time.perf_counter()-stage_started)/60 > MAX_STAGE_MINUTES:
                raise TimeoutError('Reader budget reached. Rerun this cell to resume saved questions.')
            result = ask_reader(knowledge, question)
            record = {'example_id': example['id'], 'condition': condition, 'question_index': qi,
                'question': question, 'answers': answers, 'generation': result, **score(result['text'], answers)}
            save_json(path, record)
            print(example['id'], condition, qi, repr(result['text']), record['exact_match'])
print('Question answering complete.')


## 10. Summaries, runtime estimate and export
Runtime estimates are projections from these pilot contexts, not guarantees. Memory construction happens **once per context**, not once per question. Keep SH and MH separate; ten questions are too few to establish a reliable model ranking.

“Pipeline complete” only means the files exist. Successful research also requires manually checking the logs and evaluating the reader controls. A low accuracy score is not automatically a software bug.


In [ ]:
import pandas as pd
summary = []
for example in examples:
    folder = OUTPUT / example['id']
    chunks = json.loads((folder / 'chunks.json').read_text())
    events = [json.loads((folder / f'chunk_{i:03d}.json').read_text()) for i in range(len(chunks))]
    row = {'example_id': example['id'], 'chunks': len(events),
        'write_seconds': sum(e['wall_seconds'] for e in events),
        'write_peak_allocated_GiB': max(e['generation']['peak_allocated_GiB'] for e in events),
        'write_peak_reserved_GiB': max(e['generation']['peak_reserved_GiB'] for e in events),
        'tool_calls': sum(len(e['calls']) for e in events),
        'tool_errors': sum(c['error'] for e in events for c in e['calls']),
        'chunks_no_calls': sum(e['no_tool_calls'] for e in events),
        'capped_write_responses': sum(e['generation']['hit_output_limit'] for e in events)}
    state = json.loads((folder / 'memory_final.json').read_text())
    row['memory_tokens_reader_tokenizer'] = len(reader_tokenizer.encode(memory_text(state), add_special_tokens=False))
    for condition in ['stored_memory', 'full_context_control']:
        answers = [json.loads((folder / f'answer_{condition}_{i:03d}.json').read_text()) for i in range(len(example['questions']))]
        row[condition + '_n'] = len(answers)
        for metric in ['exact_match', 'substring_exact_match']:
            row[condition + '_' + metric] = sum(a[metric] for a in answers)/len(answers)
        row[condition + '_seconds'] = sum(a['generation']['seconds'] for a in answers)
        row[condition + '_capped_answers'] = sum(a['generation']['hit_output_limit'] for a in answers)
        row[condition + '_peak_allocated_GiB'] = max(a['generation']['peak_allocated_GiB'] for a in answers)
    summary.append(row)
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUTPUT / 'summary.csv', index=False)
display(summary_df)
# Illustrative projection: the same two 6K contexts, 100 questions per context, three writer conditions.
benchmark_rows = [r for r in summary if r['example_id'] != 'synthetic_update']
projected_seconds = 3 * sum(r['write_seconds'] + 100*r['stored_memory_seconds']/r['stored_memory_n'] for r in benchmark_rows)
print('Illustrative 3-condition / 2-context / 100-question projection (GPU hours):', round(projected_seconds/3600, 2))
print('Excludes downloads, model swaps, repetitions, debugging and conflict-checking overhead.')
save_json(OUTPUT / 'completion.json', {'pipeline_complete': True, 'manual_review_required': True,
    'reader_controls_exact': sum(r['exact_match'] for r in control_results), 'reader_controls_n': len(control_results),
    'illustrative_projected_GPU_hours': projected_seconds/3600})

# List five events to inspect (three toy updates plus first/last SH updates).
review_paths = [OUTPUT / 'synthetic_update' / f'chunk_{i:03d}.json' for i in range(3)]
sh_dir = OUTPUT / 'factconsolidation_sh_6k'
sh_chunks = json.loads((sh_dir / 'chunks.json').read_text())
review_paths += [sh_dir / 'chunk_000.json', sh_dir / f'chunk_{len(sh_chunks)-1:03d}.json']
print('Manual review files:')
for path in review_paths: print(path)


In [ ]:
import shutil
archive = shutil.make_archive('/content/' + RUN_NAME, 'zip', root_dir=OUTPUT)
print('Saved in Drive:', OUTPUT)
from google.colab import files
files.download(archive)


## Week 2 check: did the chunking fix work?
- [ ] Download the run folder and run `python week_2/scripts/audit_lost_facts.py <run>/factconsolidation_sh_6k out.csv` (and `_mh_6k`).
- [ ] Missing facts should be near 0% (Week 1: 102/455 SH, 108/455 MH) and most chunks should show `cut_off False`.
- [ ] Record total write seconds from `summary.csv` (Week 1: ~1,040 s per context) to measure the runtime cost.
- [ ] Keep these settings identical for every writer compared later.

## Week 1 exit check (original)
- [ ] Record assigned GPU, memory, model/dataset/source revisions and dependency versions.
- [ ] Direct tool checks pass; synthetic updates and unrelated-fact retention have been inspected.
- [ ] Both complete 6K contexts were processed; no context was truncated.
- [ ] Reader controls, stored-memory answers and full-context controls are saved separately.
- [ ] Review no-tool-call chunks, tool errors and output-cap flags; document any unresolved failures.
- [ ] Save the output ZIP and completed Week 1 report; another member can explain one full trace.
- [ ] Decide: keep adapted 4B baseline, debug with a clear deadline, or propose the 1.7B scope change to the TA.

**If Colab disconnects:** reconnect, run setup/config/import/model cells, then rerun construction or answering. Same RUN_NAME + identical settings/revisions resumes saved work. You can skip writer loading/construction if all memory_final.json files already exist; load `examples` from all_examples.json before the reader stage. Partial downloads normally use the Hugging Face cache while the runtime survives. Drive stores results, not the large model cache.

**If GPU memory is exhausted:** first restart to remove old model allocations. Confirm only one LLM is loaded. If it fails again with the same settings, record the failure rather than silently reducing context. A new 1.7B run requires a new run name and changes what research claims you can make.

**Sources:** [Mem-α code](https://github.com/wangyu-ustc/Mem-alpha), [checkpoint](https://huggingface.co/YuWangX/Memalpha-4B), [MemoryAgentBench](https://github.com/HUST-AI-HYZ/MemoryAgentBench), [dataset](https://huggingface.co/datasets/ai-hyz/MemoryAgentBench), [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B), [MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), [4-bit loading](https://huggingface.co/docs/transformers/quantization/bitsandbytes).

Upstream code is downloaded at the pinned commits. Its licenses remain in those checkouts. No paid service is called; the notebook blocks OpenAI client construction as a safeguard.
